In [1]:
pip install gensim scikit-learn seaborn matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from gensim.models import Word2Vec
import numpy as np
import pandas as pd
import os
import json
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_distances
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
with open("Data/century_tokens.json", "r", encoding="utf-8") as f:
    century_tokens_loaded = json.load(f)

In [5]:
century_tokens_loaded.keys()

dict_keys(['5', '12', '13', '14', '15', '16', '18', '19', '20'])

In [6]:
century_5th = century_tokens_loaded['5']
century_12th = century_tokens_loaded['12']
century_13th = century_tokens_loaded['13']
century_14th = century_tokens_loaded['14']
century_15th = century_tokens_loaded['15']
century_16th = century_tokens_loaded['16']
century_18th = century_tokens_loaded['18']
century_19th = century_tokens_loaded['19']
century_20th = century_tokens_loaded['20']

In [7]:
century_5th

[['සම්යක්', 'යමු', 'බුද්ධාය'],
 ['කාම',
  'ලොක',
  'රූප',
  'ලොක',
  'අරූපලෝක',
  'යන්',
  'ලොකත්රයට',
  'ශ්රෙෂ්ඨ',
  'වූ',
  'මුනීන්ද්ර',
  'නම්',
  'වූ',
  'සම්යක්',
  'සම්බුද්ධයන්',
  'වහන්සේගේ',
  'ශ්රී',
  'පාද',
  'පද්මය',
  'භය',
  'ලොහ',
  'කුලාචාරයෙන්',
  'විනා',
  'සකසා',
  'වැඳ',
  'සකල',
  'ශාස්ත්රයෙහි',
  'පාරප්රාප්ත',
  'පණ්ඩිත',
  'වූ',
  'භෛෂජ්යාර්ථයෙහි',
  'දක්ෂ',
  'වූ',
  'වෙදුන්',
  'විසින්',
  'උතුම්',
  'වූ',
  'රස',
  'සංහිතාදි',
  'තන්ත්ර',
  'ශාස්ත්රයෙහි',
  'වුත',
  'කියන',
  'ලද',
  'විවිධං',
  'නානාප්රකාර',
  'වූ',
  'සාරාර්ථය',
  'උදුරාගෙණ',
  'සකල',
  'සත්වයන්ගේ',
  'රොග',
  'ව්යපසමන',
  'සංඛ්යාත',
  'සංග්රහව',
  'පිණිස',
  'මේ',
  'සාරාර්ථ',
  'සංග්රහ',
  'නම්',
  'ප්රකරණය',
  'සර්ථාවබොධය',
  'සඳහා',
  'කියන්නෙමි'],
 ['යනාදී',
  'අනෙකවිධ',
  'මානයෙන්',
  'දැක්වූ',
  'මේ',
  'වෛද්ය',
  'ශාස්ත්රය',
  'ප්රථම',
  'කල්පයෙහි',
  'ලෝකාර්ථය',
  'කරන්නා',
  'වූ',
  'මහ',
  'බ්රහ්මයා',
  'දක්ෂ',
  'පුජාපති',
  'හට',
  'කීහ'],
 ['දක්ෂ', 'ප්රජාපති', 'අග', 'විනීන්', '

In [8]:
print(f"""Number of sentences in 5th century: {len(century_5th)}\n
      Number of sentences in 12th century: {len(century_12th)}\n
      Number of sentences in 13th century: {len(century_13th)}\n
      Number of sentences in 14th century: {len(century_14th)}\n
      Number of sentences in 15th century: {len(century_15th)}\n
      Number of sentences in 16th century: {len(century_16th)}\n
      Number of sentences in 18th century: {len(century_18th)}\n
      Number of sentences in 19th century: {len (century_19th)}\n
      Number of sentences in 20th century: {len(century_20th)}""")

Number of sentences in 5th century: 33

      Number of sentences in 12th century: 59

      Number of sentences in 13th century: 523

      Number of sentences in 14th century: 80

      Number of sentences in 15th century: 144

      Number of sentences in 16th century: 10

      Number of sentences in 18th century: 112

      Number of sentences in 19th century: 708

      Number of sentences in 20th century: 822


In [9]:
class CleanCorpusIterator:
    def __init__(self, data):
        self.data = data
        # We pre-calculate this once so it's fast later
        self._length = sum(1 for item in self.data if isinstance(item, list) and item != "__DOC_END__")
        
    def __iter__(self):
        for item in self.data:
            if item == "__DOC_END__":
                continue
            if isinstance(item, list):
                yield item
                
    def __len__(self):
        # This fixes the TypeError
        return self._length

In [10]:
century_18th = CleanCorpusIterator(century_18th)

In [11]:
century_20th = CleanCorpusIterator(century_20th)

In [12]:
# # Exp1
# model_5th = Word2Vec(sentences=century_5th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_12th = Word2Vec(sentences=century_12th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_13th = Word2Vec(sentences=century_13th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_14th = Word2Vec(sentences=century_14th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_15th = Word2Vec(sentences=century_15th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_16th = Word2Vec(sentences=century_16th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_18th = Word2Vec(sentences=century_18th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_19th = Word2Vec(sentences=century_19th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_20th = Word2Vec(sentences=century_20th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)

In [13]:
# # Exp2
# model_5th = Word2Vec(sentences=century_5th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_12th = Word2Vec(sentences=century_12th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_13th = Word2Vec(sentences=century_13th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_14th = Word2Vec(sentences=century_14th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_15th = Word2Vec(sentences=century_15th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_16th = Word2Vec(sentences=century_16th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_18th = Word2Vec(sentences=century_18th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_19th = Word2Vec(sentences=century_19th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_20th = Word2Vec(sentences=century_20th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)

In [14]:
from gensim.models.callbacks import CallbackAny2Vec

class LossLogger(CallbackAny2Vec):
    '''Callback to log loss after each epoch.'''
    def __init__(self):
        self.epoch = 0
        self.last_loss = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        # Calculate loss for the current epoch specifically
        current_loss = loss - self.last_loss
        self.last_loss = loss
        print(f"Epoch {self.epoch} Loss: {current_loss}")
        self.epoch += 1

In [15]:
logging_callback = LossLogger()

In [16]:
model_13th = Word2Vec(sentences=century_13th, vector_size=50,
    window=5,
    min_count=1,
    sg=0,
    hs=1,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 61076.89453125
Epoch 1 Loss: 59883.14453125
Epoch 2 Loss: 60425.5703125
Epoch 3 Loss: 56693.171875
Epoch 4 Loss: 52190.5625
Epoch 5 Loss: 48694.46875
Epoch 6 Loss: 44907.15625
Epoch 7 Loss: 41855.03125
Epoch 8 Loss: 39747.25
Epoch 9 Loss: 39806.09375
Epoch 10 Loss: 39612.65625
Epoch 11 Loss: 37801.1875
Epoch 12 Loss: 37989.5625
Epoch 13 Loss: 38453.0625
Epoch 14 Loss: 38545.25
Epoch 15 Loss: 37688.3125
Epoch 16 Loss: 37026.6875
Epoch 17 Loss: 37509.875
Epoch 18 Loss: 37824.625
Epoch 19 Loss: 37382.25
Epoch 20 Loss: 36236.4375
Epoch 21 Loss: 35609.875
Epoch 22 Loss: 34997.0625
Epoch 23 Loss: 35798.1875
Epoch 24 Loss: 34810.5
Epoch 25 Loss: 33547.875
Epoch 26 Loss: 33507.375
Epoch 27 Loss: 34168.375
Epoch 28 Loss: 33860.875
Epoch 29 Loss: 31111.25
Epoch 30 Loss: 31942.0
Epoch 31 Loss: 31482.875
Epoch 32 Loss: 31063.625
Epoch 33 Loss: 30827.875
Epoch 34 Loss: 29415.125
Epoch 35 Loss: 28930.125
Epoch 36 Loss: 28180.375
Epoch 37 Loss: 27861.625
Epoch 38 Loss: 28152.75
Epoch 39

In [17]:
logging_callback = LossLogger()

In [18]:
model_14th = Word2Vec(sentences=century_14th, vector_size=50,
    window=5,
    min_count=1,
    sg=0,
    hs=1,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 18878.287109375
Epoch 1 Loss: 18746.306640625
Epoch 2 Loss: 18940.73046875
Epoch 3 Loss: 19361.54296875
Epoch 4 Loss: 19621.3984375
Epoch 5 Loss: 18818.4140625
Epoch 6 Loss: 18255.6953125
Epoch 7 Loss: 19104.46875
Epoch 8 Loss: 18433.59375
Epoch 9 Loss: 17528.828125
Epoch 10 Loss: 18110.859375
Epoch 11 Loss: 16669.3125
Epoch 12 Loss: 15960.640625
Epoch 13 Loss: 15657.796875
Epoch 14 Loss: 14759.84375
Epoch 15 Loss: 14354.8125
Epoch 16 Loss: 13572.9375
Epoch 17 Loss: 13586.65625
Epoch 18 Loss: 12801.875
Epoch 19 Loss: 12919.5
Epoch 20 Loss: 12586.59375
Epoch 21 Loss: 12314.125
Epoch 22 Loss: 11824.46875
Epoch 23 Loss: 12069.21875
Epoch 24 Loss: 11949.46875
Epoch 25 Loss: 11744.625
Epoch 26 Loss: 11690.71875
Epoch 27 Loss: 11986.15625
Epoch 28 Loss: 11789.0625
Epoch 29 Loss: 12280.0625
Epoch 30 Loss: 11932.46875
Epoch 31 Loss: 11730.875
Epoch 32 Loss: 12005.5
Epoch 33 Loss: 11953.21875
Epoch 34 Loss: 11524.71875
Epoch 35 Loss: 11937.59375
Epoch 36 Loss: 11104.125
Epoch 37 L

In [19]:
logging_callback = LossLogger()

In [20]:
model_15th = Word2Vec(sentences=century_15th, vector_size=50,
    window=5,
    min_count=1,
    sg=1,
    hs=1,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 219945.21875
Epoch 1 Loss: 206768.9375
Epoch 2 Loss: 180935.09375
Epoch 3 Loss: 151711.625
Epoch 4 Loss: 150144.9375
Epoch 5 Loss: 127747.625
Epoch 6 Loss: 124678.3125
Epoch 7 Loss: 128741.75
Epoch 8 Loss: 127205.875
Epoch 9 Loss: 129522.625
Epoch 10 Loss: 122516.5
Epoch 11 Loss: 122458.25
Epoch 12 Loss: 117122.625
Epoch 13 Loss: 122132.5
Epoch 14 Loss: 119844.375
Epoch 15 Loss: 111791.0
Epoch 16 Loss: 112286.0
Epoch 17 Loss: 108794.0
Epoch 18 Loss: 107823.75
Epoch 19 Loss: 101168.75
Epoch 20 Loss: 104476.75
Epoch 21 Loss: 96810.25
Epoch 22 Loss: 93353.25
Epoch 23 Loss: 90782.25
Epoch 24 Loss: 92934.5
Epoch 25 Loss: 92896.5
Epoch 26 Loss: 87076.25
Epoch 27 Loss: 80196.75
Epoch 28 Loss: 83198.75
Epoch 29 Loss: 81358.0
Epoch 30 Loss: 78265.0
Epoch 31 Loss: 77785.75
Epoch 32 Loss: 71067.0
Epoch 33 Loss: 73608.75
Epoch 34 Loss: 69774.5
Epoch 35 Loss: 68540.5
Epoch 36 Loss: 67903.0
Epoch 37 Loss: 67944.5
Epoch 38 Loss: 61961.5
Epoch 39 Loss: 59886.0
Epoch 40 Loss: 60291.5
Epoc

In [21]:
logging_callback = LossLogger()

In [22]:
model_18th = Word2Vec(sentences=century_18th, vector_size=50,
    window=5,
    min_count=1,
    sg=0,
    hs=1,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 12606.5263671875
Epoch 1 Loss: 12288.9755859375
Epoch 2 Loss: 12597.955078125
Epoch 3 Loss: 13580.48046875
Epoch 4 Loss: 13239.21875
Epoch 5 Loss: 12762.671875
Epoch 6 Loss: 13113.09375
Epoch 7 Loss: 12380.6171875
Epoch 8 Loss: 13136.9765625
Epoch 9 Loss: 12790.4453125
Epoch 10 Loss: 12767.8671875
Epoch 11 Loss: 12810.796875
Epoch 12 Loss: 13285.34375
Epoch 13 Loss: 12074.640625
Epoch 14 Loss: 11617.578125
Epoch 15 Loss: 11382.671875
Epoch 16 Loss: 10437.375
Epoch 17 Loss: 9703.9375
Epoch 18 Loss: 10088.5
Epoch 19 Loss: 9822.1875
Epoch 20 Loss: 9672.5
Epoch 21 Loss: 8963.703125
Epoch 22 Loss: 8956.25
Epoch 23 Loss: 8784.4375
Epoch 24 Loss: 8882.71875
Epoch 25 Loss: 9166.84375
Epoch 26 Loss: 8479.46875
Epoch 27 Loss: 7971.875
Epoch 28 Loss: 8611.9375
Epoch 29 Loss: 8750.46875
Epoch 30 Loss: 8160.8125
Epoch 31 Loss: 8754.65625
Epoch 32 Loss: 8182.3125
Epoch 33 Loss: 7862.78125
Epoch 34 Loss: 7803.28125
Epoch 35 Loss: 8373.21875
Epoch 36 Loss: 8023.65625
Epoch 37 Loss: 8270.

In [23]:
logging_callback = LossLogger()

In [24]:
model_19th = Word2Vec(sentences=century_19th, vector_size=50,
    window=5,
    min_count=1,
    sg=0,
    hs=1,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 56789.01953125
Epoch 1 Loss: 57515.95703125
Epoch 2 Loss: 55347.8203125
Epoch 3 Loss: 54454.703125
Epoch 4 Loss: 52009.59375
Epoch 5 Loss: 47851.3125
Epoch 6 Loss: 45253.875
Epoch 7 Loss: 42918.71875
Epoch 8 Loss: 40870.5625
Epoch 9 Loss: 39366.5625
Epoch 10 Loss: 38819.8125
Epoch 11 Loss: 38405.875
Epoch 12 Loss: 37560.4375
Epoch 13 Loss: 37865.3125
Epoch 14 Loss: 38047.25
Epoch 15 Loss: 37285.9375
Epoch 16 Loss: 36400.875
Epoch 17 Loss: 36907.1875
Epoch 18 Loss: 36532.0625
Epoch 19 Loss: 36241.5
Epoch 20 Loss: 35595.25
Epoch 21 Loss: 34903.8125
Epoch 22 Loss: 35776.625
Epoch 23 Loss: 34146.6875
Epoch 24 Loss: 34652.1875
Epoch 25 Loss: 34977.1875
Epoch 26 Loss: 34312.75
Epoch 27 Loss: 34251.25
Epoch 28 Loss: 34576.75
Epoch 29 Loss: 33562.5
Epoch 30 Loss: 34629.0
Epoch 31 Loss: 33561.5
Epoch 32 Loss: 32828.75
Epoch 33 Loss: 32181.25
Epoch 34 Loss: 32233.875
Epoch 35 Loss: 31799.5
Epoch 36 Loss: 31784.5
Epoch 37 Loss: 31590.5
Epoch 38 Loss: 31106.25
Epoch 39 Loss: 30671.75

In [25]:
logging_callback = LossLogger()

In [26]:
model_20th = Word2Vec(sentences=century_20th, vector_size=50,
    window=5,
    min_count=1,
    sg=0,
    hs=1,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 69231.2265625
Epoch 1 Loss: 69710.8828125
Epoch 2 Loss: 69761.234375
Epoch 3 Loss: 67986.84375
Epoch 4 Loss: 66182.34375
Epoch 5 Loss: 58387.34375
Epoch 6 Loss: 55391.15625
Epoch 7 Loss: 50662.96875
Epoch 8 Loss: 46955.875
Epoch 9 Loss: 46421.6875
Epoch 10 Loss: 45331.875
Epoch 11 Loss: 43161.375
Epoch 12 Loss: 44831.9375
Epoch 13 Loss: 42658.625
Epoch 14 Loss: 42665.5625
Epoch 15 Loss: 42568.875
Epoch 16 Loss: 42386.9375
Epoch 17 Loss: 41945.25
Epoch 18 Loss: 42105.875
Epoch 19 Loss: 40972.5625
Epoch 20 Loss: 39663.9375
Epoch 21 Loss: 39625.875
Epoch 22 Loss: 38917.25
Epoch 23 Loss: 38233.125
Epoch 24 Loss: 38386.25
Epoch 25 Loss: 37803.375
Epoch 26 Loss: 37023.5
Epoch 27 Loss: 35662.75
Epoch 28 Loss: 35943.75
Epoch 29 Loss: 35484.75
Epoch 30 Loss: 34141.125
Epoch 31 Loss: 33936.625
Epoch 32 Loss: 32531.75
Epoch 33 Loss: 32187.75
Epoch 34 Loss: 32769.0
Epoch 35 Loss: 30820.875
Epoch 36 Loss: 30893.625
Epoch 37 Loss: 29774.625
Epoch 38 Loss: 29471.125
Epoch 39 Loss: 28973

In [27]:
logging_callback = LossLogger()

In [28]:
model_13th.save("Word2Vec/Exp4/model_13th.model")
model_14th.save("Word2Vec/Exp4/model_14th.model")
model_15th.save("Word2Vec/Exp4/model_15th.model")
model_18th.save("Word2Vec/Exp4/model_18th.model")
model_19th.save("Word2Vec/Exp4/model_19th.model")
model_20th.save("Word2Vec/Exp4/model_20th.model")

In [29]:
# all_centuries_data = []
# for century in century_tokens_loaded.values():
#     all_centuries_data.extend(century)

In [30]:
# all_centuries_data = CleanCorpusIterator(all_centuries_data)

In [31]:
# base_model = Word2Vec(
#     vector_size=50,
#     window=8,
#     min_count=1,
#     sg=1,
#     hs=0,
#     negative=20,
#     sample=1e-2,
#     alpha=0.03,
#     min_alpha=0.0007,
#     workers=4,
#     seed=42
# )

# # 2. Build the Vocabulary (Tells the model which words exist)
# base_model.build_vocab(all_centuries_data)

# # 3. Train the Model (Learn the relationships)
# base_model.train(
#     all_centuries_data, 
#     total_examples=base_model.corpus_count, 
#     epochs=50 # Use the full 50 here
# )

In [32]:
# print(f"Total words in vocabulary: {len(base_model.wv)}")
# # Check a few neighbors to see if it makes sense for your dataset
# print(base_model.wv.most_similar(positive=['රජ'], topn=5))

In [33]:
print(f"Total words in vocabulary: {len(model_20th.wv)}")
# Check a few neighbors to see if it makes sense for your dataset
print(model_20th.wv.most_similar(positive=['රජ'], topn=5))

Total words in vocabulary: 7993
[('නඋන්නාන්සේ', 0.6312391757965088), ('බුද්ධාගමය', 0.624707043170929), ('මහ', 0.6092764735221863), ('උන්නෙය', 0.6050759553909302), ('වවන්නේ', 0.6012358069419861)]


In [34]:
# base_model.train(century_20th, total_examples=len(century_20th), epochs=100)

In [35]:
# model_20th.train(
#     century_20th, 
#     total_examples=len(century_20th), 
#     epochs=50 # Moderate fine-tuning
# )

In [36]:
# model_base = Word2Vec(
#     sentences=all_centuries_data, 
#     vector_size=100, 
#     window=5, 
#     min_count=1, 
#     sg=1, 
#     workers=4, 
#     epochs=20 # Enough to learn general structure
# )
# model_base.save("base_foundation.model")

In [37]:
centuries = ['13th', '14th', '15th', '18th', '19th', '20th']

In [38]:
import io
import os

# Ensure the directory exists
os.makedirs('Word2Vec/Embeddings', exist_ok=True)

for c_suffix in centuries:

    model_variable_name = f'model_{c_suffix}'
    
    if model_variable_name not in globals():
        print(f"Skipping {model_variable_name}: Model object not found.")
        continue
        
    current_model = globals()[model_variable_name]
    
    save_path = f'Word2Vec/Embeddings/{c_suffix}'
    os.makedirs(save_path, exist_ok=True)

    out_v = io.open(f'{save_path}/vectors.tsv', 'w', encoding='utf-8')
    out_m = io.open(f'{save_path}/metadata.tsv', 'w', encoding='utf-8')

    current_model.wv.fill_norms() 
    words = current_model.wv.index_to_key

    for word in words:
        vec = current_model.wv.get_vector(word, norm=True)
        
        out_v.write('\t'.join([str(x) for x in vec]) + "\n")
        out_m.write(word + "\n")

    out_v.close()
    out_m.close()
    print(f"Successfully exported {model_variable_name}")

Successfully exported model_13th
Successfully exported model_14th
Successfully exported model_15th
Successfully exported model_18th
Successfully exported model_19th
Successfully exported model_20th
